# Axis 2 — Cosmos-Predict2-V2W pilot (heavily optimized)

Single-purpose notebook. Probes verb-spatial binding in `nvidia/Cosmos-Predict2-2B-Video2World` on AGD20K. Designed to be **GPU-bound, not IO-bound**.

## Optimizations vs the unified notebook

| Bottleneck | What we did | Why |
|---|---|---|
| Drive FUSE on every CSV append | Results live on `/content/cosmos_results` (local SSD) | Drive is REST-mounted; per-row writes stall over hours |
| `.npy` map per sample → 1000 Drive writes | `--save_attention_maps` OFF | Attention can be re-extracted later for qualitative panel |
| Synchronous `git push` every N samples | Replaced with background `rsync` to Drive | git push blocks the cell; rsync runs in a daemon thread |
| Diffusers inner progress bar (1 print per step) | `set_progress_bar_config(disable=True)` | ~1 s per sample over 1000 samples = 15 min wasted |
| Cosmos default 480×704 × 17 frames × 12 steps | Reduced to 384×384 × 9 frames × 8 steps | 3-4× fewer attention ops per sample; cross-attention pattern is stable at this scale |
| HF cache on ephemeral runtime disk | Symlinked to Drive (one-time, then reused) | Model download survives disconnects |
| No resume logic across session deaths | Local CSV is rsync'd to Drive every 60s; on session restart Drive copy is rehydrated to local | Lets a dead session pick up where it left off |

## Expected timing

~15-20 s per sample on A100. With max_per_category=30 across 36 categories minus skips, ~1000 samples → **~4-5 hours**. Fits one Colab Pro session.

## What is NOT in this notebook

- Flux runs (the unified notebook handles those — let that finish in parallel if it's still going).
- Cosmos Policy (deferred — not a diffusers pipeline).
- Three-way comparison (`scripts/12_three_way_comparison.py` runs locally after both pilots finish).

## Cell 0 — Repo + dependencies (silenced)

Silences transformers/diffusers/accelerate logging *before* any model touches a logger. Sets `HF_HOME` to Drive so the 14 GB Cosmos download survives disconnects.

In [ ]:
import os, subprocess, logging
from pathlib import Path

# Silence the noisy loggers *before* anything else imports them.
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['DIFFUSERS_VERBOSITY'] = 'error'
os.environ['ACCELERATE_LOG_LEVEL'] = 'ERROR'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
for _noisy in ('transformers', 'diffusers', 'accelerate', 'huggingface_hub'):
    logging.getLogger(_noisy).setLevel(logging.ERROR)

REPO = 'aleksantari/VLA-affordance'
BRANCH = 'nj-features'
REPO_DIR = '/content/VLA-affordance'

GH_PAT = ''
try:
    from google.colab import userdata
    GH_PAT = userdata.get('GH_PAT') or ''
    print('GH_PAT' + (' found' if GH_PAT else ' missing — git push will skip'))
except Exception:
    print('Colab userdata unavailable')

remote = f'https://{GH_PAT}@github.com/{REPO}.git' if GH_PAT else f'https://github.com/{REPO}.git'
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', remote, REPO_DIR], check=True)
%cd /content/VLA-affordance
subprocess.run(['git', 'remote', 'set-url', 'origin', remote], check=True)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
subprocess.run(['git', 'config', 'user.email', 'jajda.n@northeastern.edu'], check=True)
subprocess.run(['git', 'config', 'user.name', 'nitik1998 (Colab)'], check=True)

!pip install -e . -q
!pip install -r requirements_axis2.txt -q
print('\nsetup complete')

## Cell 1 — GPU + HF auth + persistent HF cache on Drive

In [ ]:
import torch, os
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type > A100')
name = torch.cuda.get_device_name()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  VRAM: {vram:.1f} GB')
if vram < 35:
    raise RuntimeError(f'Cosmos V2W needs ~32 GB; this GPU has {vram:.1f} GB. Pick a stronger A100/H100.')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_HF_CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = DRIVE_HF_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE + '/hub'
os.environ['TRANSFORMERS_CACHE'] = DRIVE_HF_CACHE + '/transformers'
print(f'HF_HOME = {DRIVE_HF_CACHE}')

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('HF login ok')
    else:
        print('No HF_TOKEN in Colab secrets — gated models will fail')
except Exception as e:
    print(f'HF login skipped: {e}')

## Cell 2 — Local results dir + AGD20K + Drive rehydrate

Results dir lives on local SSD (fast). If a previous session pushed a CSV to Drive, we copy it back so resume picks up where we left off.

In [ ]:
import shutil, os
from pathlib import Path

LOCAL_RESULTS = Path('/content/cosmos_results')          # FAST local disk
DRIVE_RESULTS = Path('/content/drive/MyDrive/VLA-affordance-results/cosmos')  # Persistent
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

# Rehydrate: if Drive has a CSV from a prior session, copy back to local
drive_csv = DRIVE_RESULTS / 'tables' / 'axis2_cosmos_predict2_v2w_per_sample.csv'
local_csv = LOCAL_RESULTS / 'tables' / 'axis2_cosmos_predict2_v2w_per_sample.csv'
if drive_csv.exists() and not local_csv.exists():
    local_csv.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(drive_csv), str(local_csv))
    n = sum(1 for _ in open(local_csv)) - 1
    print(f'Rehydrated {n} prior samples from Drive')
else:
    print(f'Starting fresh (no prior Drive CSV at {drive_csv})')

# Point ./results at the local dir (NOT a Drive symlink)
if Path('./results').exists() or Path('./results').is_symlink():
    if Path('./results').is_symlink():
        os.unlink('./results')
    else:
        shutil.rmtree('./results')
os.symlink(str(LOCAL_RESULTS), './results')
print(f'./results -> {LOCAL_RESULTS} (LOCAL, fast)')

# AGD20K — same logic as the unified notebook
DRIVE_AGD20K_DIR = Path('/content/drive/MyDrive/datasets/AGD20K')
DRIVE_AGD20K_ZIP = Path('/content/drive/MyDrive/LBV Project/AGD20K.zip')

if DRIVE_AGD20K_DIR.exists() and any(DRIVE_AGD20K_DIR.iterdir()):
    print(f'Found unzipped AGD20K at {DRIVE_AGD20K_DIR}')
    !python data/download_agd20k.py --from_drive "{DRIVE_AGD20K_DIR}" --output_dir ./data/agd20k
elif DRIVE_AGD20K_ZIP.exists():
    print(f'Unzipping {DRIVE_AGD20K_ZIP} (one-time)')
    !python data/download_agd20k.py --from_zip "{DRIVE_AGD20K_ZIP}" --output_dir ./data/agd20k
    DRIVE_AGD20K_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree('./data/agd20k', str(DRIVE_AGD20K_DIR), dirs_exist_ok=True)
    print(f'Persisted to {DRIVE_AGD20K_DIR}')
else:
    print('Falling back to direct download (LOCATE mirror)')
    !python data/download_agd20k.py --output_dir ./data/agd20k

!python data/download_agd20k.py --validate_only --output_dir ./data/agd20k

## Cell 3 — Background Drive sync

Daemon thread that rsync's the CSV every 60 s and the full results dir every 5 min. Sync is async and never blocks the main cell. If you Ctrl-C the run, daemon dies cleanly.

In [ ]:
import threading, time, subprocess

_sync_stop = threading.Event()

def _bg_sync():
    iteration = 0
    while not _sync_stop.is_set():
        try:
            # Every iteration (60 s): sync just the CSV — small, frequent
            subprocess.run(
                ['rsync', '-a',
                 f'{LOCAL_RESULTS}/tables/',
                 f'{DRIVE_RESULTS}/tables/'],
                check=False, capture_output=True, timeout=30,
            )
            # Every 5 iterations (5 min): full results dir sync
            if iteration % 5 == 0:
                subprocess.run(
                    ['rsync', '-a', f'{LOCAL_RESULTS}/', f'{DRIVE_RESULTS}/'],
                    check=False, capture_output=True, timeout=120,
                )
        except Exception as e:
            pass
        iteration += 1
        _sync_stop.wait(timeout=60)

_sync_thread = threading.Thread(target=_bg_sync, daemon=True)
_sync_thread.start()
print(f'Background sync armed: CSV every 60 s, full dir every 5 min, → {DRIVE_RESULTS}')

## Cell 4 — Pre-flight diagnostic (5 sec, no GPU)

Verifies imports, AGD20K detection, GT-heatmap finder. Saves you a 5-minute pipeline load if anything's broken.

In [ ]:
!python scripts/00_diagnostic.py

## Cell 5 — Smoke test (~3 min)

Loads Cosmos pipeline once, runs ONE inference at tiny resolution, verifies the custom AttentionProcessor records something non-trivial. If this fails, do NOT run the pilot — fix locally, push, re-pull.

In [ ]:
!python scripts/09b_setup_cosmos.py --system cosmos_predict2_v2w \
    --num_inference_steps 4 --num_frames 5 --height 320 --width 320

## Cell 6 — Mini-pilot (~5-8 min)

3 categories × 3 AGD20K samples each. Confirms the dataset/extractor handshake works end-to-end on real data before committing to the full pilot.

In [ ]:
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_predict2_v2w \
    --categories cut hold pour \
    --max_per_category 3 \
    --num_inference_steps 6 \
    --num_frames 5 \
    --height 320 --width 320 \
    --output_dir /content/cosmos_results

## Cell 7 — Full pilot (the main run, ~4-5 hrs)

**Optimized Cosmos config:**
- `num_inference_steps = 8` — cross-attention pattern stabilises by step 4-6
- `num_frames = 9` — minimum for stable video latent; we average over temporal
- `height = 384, width = 384` — square, ~half the spatial tokens of 480×704
- `max_per_category = 30` — matches the Flux pilot for paired comparison
- **No `--save_attention_maps`** — saves Drive thrash
- **No `--commit_every`** — background rsync handles persistence
- `--resume` is default ON; CSV at local /content/cosmos_results/tables/ feeds resume

In [ ]:
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_predict2_v2w \
    --max_per_category 30 \
    --num_inference_steps 8 \
    --num_frames 9 \
    --height 384 --width 384 \
    --output_dir /content/cosmos_results

## Cell 8 — Final sync + git push

After Cell 7 completes (or if you interrupt), do one final full rsync to Drive and push the CSV to GitHub so the agent can pull and analyze locally.

In [ ]:
# Stop background sync and force one final flush
_sync_stop.set()
import subprocess
subprocess.run(['rsync', '-a', f'{LOCAL_RESULTS}/', f'{DRIVE_RESULTS}/'],
               check=False)
print(f'Final Drive sync done: {DRIVE_RESULTS}')

# Push the CSV to the repo so the agent can analyze locally
!cp -r /content/cosmos_results/tables /content/VLA-affordance/results/
%cd /content/VLA-affordance
!git add results/tables
!git status -sb
!git commit -m 'results(cosmos_v2w): pilot complete — optimized config' || echo 'nothing new to commit'
!git push origin nj-features || echo 'push failed; results still on Drive at {DRIVE_RESULTS}'

## Cell 9 — (Optional) Idle keepalive

Fires a JS heartbeat every 30 s so Colab doesn't disconnect during the ~5 hr pilot. Run this in parallel with Cell 7. Doesn't bypass the absolute session cap.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ColabKeepalive() {
  document.querySelector('colab-toolbar-button#connect')?.click();
}
setInterval(ColabKeepalive, 30000);
'''))
print('Keepalive armed (every 30 s)')